## Source-only notebook copy

All saved outputs, embedded media/attachments, execution counts and environment-specific metadata have been removed from this copy. Historical results and limitations are documented in README.md; the original academic submission is unchanged. No project cells were executed during packaging. Input images, datasets, model weights and historical result files are not included.


## **Enhancing Quality and Efficiency: A study on Image Compression and Quality Retention**

This notebook evaluates the performance of different image compression formats (JPEG, PNG, WEBP) by comparing their:
- Compressed file size
- Compression time
- Quality loss using PSNR and SSIM metrics

### Import Required Libraries

In [ ]:
import os
import cv2
import time
import numpy as np
import pandas as pd
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim

### Utility Function

In [ ]:
# Returns file size in bytes for a given file path
def get_file_size(file_path):
    return os.path.getsize(file_path)

### Compress Image with Format and Quality Settings

In [ ]:
# Compresses an image to a specified format and quality level
def compress_image(input_path, output_path, format, quality=None):
    img = Image.open(input_path)

    # Convert to RGB if format requires it (JPEG or WebP)
    if format.upper() in ['JPEG', 'WEBP'] and img.mode != 'RGB':
        img = img.convert('RGB')

    save_params = {}

    # Set quality and compression method parameters
    if format.upper() in ['JPEG', 'WEBP'] and quality is not None:
        save_params['quality'] = quality
        if format.upper() == 'WEBP':
            save_params['method'] = 6  

    # Set PNG compression level
    if format.upper() == 'PNG':
        save_params['compress_level'] = 9

    # Save the compressed image and measure the time taken
    start_time = time.time()
    img.save(output_path, format=format, **save_params)
    return time.time() - start_time

### Evaluate Compression (PSNR, SSIM, Size, Time)

In [ ]:
# Evaluates compression by measuring PSNR, SSIM, file size, and time taken.
def evaluate_compression(image_path, format, quality_levels):
    img_pil_orig = Image.open(image_path)
    img_cv2_orig = cv2.cvtColor(np.array(img_pil_orig.convert("RGB")), cv2.COLOR_RGB2BGR)
    original_size = get_file_size(image_path)

    results = []

    for quality in quality_levels:
        ext = "jpg" if format.upper() == "JPEG" else format.lower()
        quality_suffix = f"_q{quality}" if quality is not None else ""
        out_file = f"temp_compressed_{format.lower()}{quality_suffix}.{ext}"

        compress_time = compress_image(image_path, out_file, format, quality)
        compressed_size = get_file_size(out_file)

        img_cv2_comp = cv2.cvtColor(np.array(Image.open(out_file).convert("RGB")), cv2.COLOR_RGB2BGR)

        if np.array_equal(img_cv2_orig, img_cv2_comp):
            value_psnr = float('inf')
        else:
            value_psnr = psnr(img_cv2_orig, img_cv2_comp, data_range=255)

        value_ssim = ssim(img_cv2_orig, img_cv2_comp, channel_axis=2)

        results.append({
            "Image": os.path.basename(image_path),
            "Format": format,
            "Quality": quality,
            "Original Size (KB)": round(original_size / 1024, 2),
            "Compressed Size (KB)": round(compressed_size / 1024, 2),
            "Compression Time (s)": round(compress_time, 4),
            "PSNR": round(value_psnr, 2) if value_psnr != float('inf') else float('inf'),
            "SSIM": round(value_ssim, 4)
        })

        os.remove(out_file)
        
    return results

### Process Entire Dataset Folder - Images 

In [ ]:
# Walks through image dataset folder and evaluates each image with multiple formats.
def process_dataset_folder(dataset_root, quality_levels):
    all_results = []

    for category in os.listdir(dataset_root):
        category_path = os.path.join(dataset_root, category)

        if not os.path.isdir(category_path):
            continue

        for filename in os.listdir(category_path):
            if filename.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                image_path = os.path.join(category_path, filename)

                for fmt in ["JPEG", "WEBP", "PNG"]:
                    q_levels = quality_levels if fmt != "PNG" else [None]
                    results = evaluate_compression(image_path, fmt, q_levels)
                    for res in results:
                        res["Category"] = category
                        all_results.append(res)

    return all_results


### Save compressed images

In [ ]:
# Compress and save the input image in PNG, JPEG, and WEBP formats
def compress_image_all_formats(input_path, output_dir, base_name, quality=50):
    os.makedirs(output_dir, exist_ok=True)
    
    formats = {
        "PNG": {"ext": "png", "quality": None},
        "JPEG": {"ext": "jpg", "quality": quality},
        "WEBP": {"ext": "webp", "quality": quality}
    }
    
    saved_paths = {}
    
    for fmt, params in formats.items():
        quality_suffix = f"_q{quality}" if params["quality"] is not None else ""
        filename = f"{base_name}{quality_suffix}.{params['ext']}"
        out_path = os.path.join(output_dir, filename)
        
        # Compress and save
        compress_image(input_path, out_path, format=fmt, quality=params["quality"])
        
        saved_paths[fmt] = out_path
    
    return saved_paths

output_dir = "compressed"
quality = 50

image_list = {
    "graphics": "images/graphics/g_1.png",
    "highresol": "images/high-resolution/hr_1.png",
    "photographs": "images/photographs/p_1.png",
    "textbased": "images/text-based/tb_1.png"
}

for category, original_path in image_list.items():
    base_name = category  # use category name as base filename
    compress_image_all_formats(original_path, output_dir, base_name, quality=quality)


### Main Execution: Run and Save Results

In [ ]:
import os

# Set dataset path and quality levels for testing
dataset_folder_path = "images"  
quality_levels = [90, 70, 50]  

# Run the full evaluation process (make sure this compresses but deletes temp files)
results = process_dataset_folder(dataset_folder_path, quality_levels)

# Convert results to DataFrame
df = pd.DataFrame(results)

# Display the results
print("\nCompression Results Table:")
print(df.to_string(index=False))

# Save results to CSV
df.to_csv("compression_results.csv", index=False)
print("\nResults saved to 'compression_results.csv'.")

### Summarize Compression Metrics

In [ ]:
# Summarizes average PSNR, SSIM, and other metrics per format/quality/category.
def summarize_results(csv_path="compression_results.csv"):
    df = pd.read_csv(csv_path)

    # Make sure Format and Quality columns are consistent
    df["Format"] = df["Format"].str.lower().str.strip()
    
    # Convert Quality to numeric (some formats like png might have NaNs here)
    df["Quality"] = pd.to_numeric(df["Quality"], errors='coerce')
    
    # Drop rows with missing essential info (Quality or Format)
    df = df.dropna(subset=["Format", "Quality"])

    # Average by format and quality
    summary_format_quality = df.groupby(["Format", "Quality"]).agg({
        "PSNR": "mean",
        "SSIM": "mean",
        "Original Size (KB)": "mean",
        "Compressed Size (KB)": "mean",
        "Compression Time (s)": "mean"
    }).reset_index()

    # Average by category and format (if 'Category' exists)
    if "Category" in df.columns:
        summary_category = df.groupby(["Category", "Format"]).agg({
            "PSNR": "mean",
            "SSIM": "mean"
        }).reset_index()
    else:
        summary_category = pd.DataFrame()  # empty if no Category column

    return summary_format_quality, summary_category


# Usage
summary_fq, summary_cat = summarize_results("compression_results.csv")

print(summary_fq.head())  # Check your clean summary by format and quality
print(summary_cat.head()) # Check your category summary (if available)



## **Visualize the Results**

### Line graphs to illustrate PSNR and SSIM trends at different compression levels

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot SSIM and PSNR comparisons side by side
def plot_metrics(summary_df):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # SSIM plot
    sns.lineplot(data=summary_df, x="Quality", y="SSIM", hue="Format", marker="o", ax=axes[0])
    axes[0].set_title("SSIM vs. Quality Level")
    axes[0].set_ylabel("SSIM")
    axes[0].set_xlabel("Quality Level")
    axes[0].grid(True)

    # PSNR plot
    sns.lineplot(data=summary_df, x="Quality", y="PSNR", hue="Format", marker="o", ax=axes[1])
    axes[1].set_title("PSNR vs. Quality Level")
    axes[1].set_ylabel("PSNR (dB)")
    axes[1].set_xlabel("Quality Level")
    axes[1].grid(True)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

plot_metrics(summary_fq)


### Visual Comparison of Original vs Compressed Image

In [ ]:
def show_comparison(original_path, compressed_paths):
    # Load images
    orig = Image.open(original_path)
    comp_png = Image.open(compressed_paths["PNG"])
    comp_jpeg = Image.open(compressed_paths["JPEG"])
    comp_webp = Image.open(compressed_paths["WEBP"])

    images = [orig, comp_png, comp_jpeg, comp_webp]
    titles = ["Original", "Compressed PNG", "Compressed JPEG", "Compressed WEBP"]

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

compressed_paths = {
    "PNG": "compressed/highresol.png",
    "JPEG": "compressed/highresol_q50.jpg",
    "WEBP": "compressed/highresol_q50.webp"
}

show_comparison("images/high-resolution/hr_1.png", compressed_paths)

### Heatmaps to highlight compression and decompression speeds.

In [ ]:
# Assume compress_image(input_path, output_path, format, quality=None) is defined
image_list = {
    "graphics": "images/graphics/g_1.png",
    "highresol": "images/high-resolution/hr_1.png",
    "photographs": "images/photographs/p_1.png",
    "textbased": "images/text-based/tb_1.png"
}

output_dir = "compressed"
quality = 50
formats = {
    "PNG": {"ext": "png", "quality": None},
    "JPEG": {"ext": "jpg", "quality": quality},
    "WEBP": {"ext": "webp", "quality": quality}
}

def timed_compress_image(input_path, output_path, format, quality=None):
    start = time.time()
    compress_image(input_path, output_path, format=format, quality=quality)
    end = time.time()
    return end - start

def timed_decompress_image(path):
    start = time.time()
    img = Image.open(path)
    img.load()  # force read/decompression
    end = time.time()
    return end - start

# Store times here
compress_times = {fmt: [] for fmt in formats.keys()}
decompress_times = {fmt: [] for fmt in formats.keys()}

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

for category, input_path in image_list.items():
    for fmt, params in formats.items():
        # Compose output filename
        if fmt == "PNG":
            filename = f"{category}.{params['ext']}"
        else:
            filename = f"{category}_q{quality}.{params['ext']}"
        output_path = os.path.join(output_dir, filename)
        
        # Compress and time
        c_time = timed_compress_image(input_path, output_path, fmt, params["quality"])
        compress_times[fmt].append(c_time)
        
        # Decompress and time
        d_time = timed_decompress_image(output_path)
        decompress_times[fmt].append(d_time)

# Create DataFrames for plotting
categories = list(image_list.keys())
df_compress = pd.DataFrame(compress_times, index=categories)
df_decompress = pd.DataFrame(decompress_times, index=categories)

# Plot heatmaps
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.heatmap(df_compress, annot=True, fmt=".4f", cmap="YlGnBu", cbar_kws={'label': 'Seconds'})
plt.title("Compression Time Heatmap")
plt.xlabel("Format")
plt.ylabel("Image Category")

plt.subplot(1, 2, 2)
sns.heatmap(df_decompress, annot=True, fmt=".4f", cmap="YlOrRd", cbar_kws={'label': 'Seconds'})
plt.title("Decompression Time Heatmap")
plt.xlabel("Format")
plt.ylabel("Image Category")

plt.tight_layout()
plt.show()


### Bar charts to show file size reduction

In [ ]:
# Collect sizes here
original_sizes = []
compressed_sizes = {fmt: [] for fmt in formats.keys()}

for category, input_path in image_list.items():
    # Get original file size in KB
    original_size = os.path.getsize(input_path) / 1024
    original_sizes.append(original_size)
    
    for fmt, params in formats.items():
        if fmt == "PNG":
            filename = f"{category}.{params['ext']}"
        else:
            filename = f"{category}_q{quality}.{params['ext']}"
        compressed_path = os.path.join(output_dir, filename)
        
        # Get compressed file size in KB
        if os.path.exists(compressed_path):
            size = os.path.getsize(compressed_path) / 1024
        else:
            size = 0  # Or handle missing file differently
        compressed_sizes[fmt].append(size)

# Compute reduction percentage = ((orig - compressed) / orig) * 100
reduction = {}
for fmt in formats.keys():
    reduction[fmt] = [
        100 * (o - c) / o if o > 0 else 0
        for o, c in zip(original_sizes, compressed_sizes[fmt])
    ]

categories = list(image_list.keys())
x = np.arange(len(categories))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))

bars1 = ax.bar(x - width, reduction["PNG"], width, label="PNG")
bars2 = ax.bar(x, reduction["JPEG"], width, label="JPEG")
bars3 = ax.bar(x + width, reduction["WEBP"], width, label="WEBP")

ax.set_ylabel("File Size Reduction (%)")
ax.set_title("File Size Reduction by Image Category and Format")
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()

# Show value labels on bars
def autolabel(bars):
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.1f}%',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom')

autolabel(bars1)
autolabel(bars2)
autolabel(bars3)

plt.tight_layout()
plt.show()
